# Radiance Observation Simulator
This notebook execises the classes that implement the JEDI observation simulator for radiances. 

In [1]:
import numpy  as np
import xarray as xr

from datetime        import datetime, timedelta
from IPython.display import Markdown, display

from obsimul import radiance as rd

import matplotlib.pyplot as plt
%matplotlib inline

## Geophysical Variable Simulator
Takes output from model sampler and generates as it is needed by the UFO tester.

In [2]:
def getFiles(sensor, time):
    """
    Function to generate radiance geovals for a given sensor and date.

    Parameters:
    - sensor (str): The sensor identifier.
    - time: time for sampling file.
    """
    year, month, day, hour = time.year, time.month, time.day, time.hour
    
    print(f"Processing {sensor} for {year}-{month:02d}-{day:02d} at {hour:02d}Z")

    # Directory and filename for input and output data
    nr_dirn = '/discover/nobackup/projects/gmao/aist-nr/yyu11/run_geos_su17/c180_L137_test_sampler/scratch.jedi.works/'
    nr_file = nr_dirn+f"{sensor}.AIST_c180_L137.jedi.{year}{month:02d}{day:02d}_{hour:02d}00z.nc4"


    sfc_dirn = '/discover/nobackup/projects/gmao/nwposse/mkim1/AIST-NR/jedi-ncep-sfc-files/veg20/'
    sfc_file = sfc_dirn +'jedi.crtmsrf.1152x721.nc4'

    gv_dirn = './'
    gv_file = gv_dirn+f"{sensor}_geovals.{year}{month:02d}{day:02d}T{hour:02d}0000Z.nc4"

    return (nr_file, sfc_file, gv_file)

def fixNR(nr):
    """
    Fix structure of old sampling files. This a place holder until new files
    are available.
    """

    alias = dict ( dateTime = 'time',
                    sphu = 'QV',
                   qitot = 'QI',
                   qltot = 'QL',
                      tv = 'T',   # not the same thing, but good enough
                   ozone = 'O3',
                 )

    for v in ('phis', 'hs_stdv', 'ts', 'frland', 'frlandice', 'frlake', 'frocean', 
              'frseaice', 'ps', 'delp', 'u', 'v' ):
        alias[v] = v.upper()

    nr = nr.rename(alias)

    # Place Holders
    # -------------
    nr['QR']  = 0 * nr.QL
    nr['QS']  = 0 * nr.QL
    nr['CO2'] = 0 * nr.O3 + 410
    nr['LAI'] = 0 * nr.QL 

    nlev, nlocs = nr.DELP.shape
    nr['PLE'] = xr.DataArray(np.ones((nlev+1,nlocs)).astype('float32'),
                             dims=['levp1','Location'],
                             attrs = {'long_name':'Pressure at Edges', 'units':'pa' })

    for k in range(nlev):
        nr.PLE[k+1] = nr.PLE[k] + nr.DELP[k]
    
    # Fix order of dimensions for profiles
    for v in nr.data_vars:
        if len(nr[v].shape) == 2:
            nr[v] = nr[v].T
    
    return nr


In [3]:
# Generate file names
# -------------------
time = datetime(2019,8,1,3)
nr_file, sfc_file, gv_file = getFiles('atms_n20',time)
print('[] NR  file',  nr_file)
print('[] SFC file', sfc_file)
print('[] GV  file',  gv_file)

Processing atms_n20 for 2019-08-01 at 03Z
[] NR  file /discover/nobackup/projects/gmao/aist-nr/yyu11/run_geos_su17/c180_L137_test_sampler/scratch.jedi.works/atms_n20.AIST_c180_L137.jedi.20190801_0300z.nc4
[] SFC file /discover/nobackup/projects/gmao/nwposse/mkim1/AIST-NR/jedi-ncep-sfc-files/veg20/jedi.crtmsrf.1152x721.nc4
[] GV  file ./atms_n20_geovals.20190801T030000Z.nc4


In [4]:
# Lazy-load Nature Run file
# -------------------------
nr = xr.open_dataset(nr_file,engine='netcdf4')
display(Markdown('### OLD Sampled Nature Run'), nr)

nr = fixNR(nr)
display(Markdown('### NEW Sampled Nature Run'), nr)

# Lazy-load GF surface file
# -------------------------
sfc = xr.open_dataset(sfc_file,engine='netcdf4')

display(Markdown('### GFS Surface'), sfc)


### OLD Sampled Nature Run

<xarray.Dataset> Size: 49MB
Dimensions:    (lev: 137, Location: 10578)
Coordinates:
  * lev        (lev) float64 1kB 1.0 2.0 3.0 4.0 5.0 ... 134.0 135.0 136.0 137.0
Dimensions without coordinates: Location
Data variables: (12/63)
    dateTime   (Location) datetime64[ns] 85kB ...
    longitude  (Location) float64 85kB ...
    latitude   (Location) float64 85kB ...
    phis       (Location) float32 42kB ...
    hs_stdv    (Location) float32 42kB ...
    ts         (Location) float32 42kB ...
    ...         ...
    Z0M        (Location) float32 42kB ...
    DCOOL      (Location) float32 42kB ...
    DWARM      (Location) float32 42kB ...
    TDROP      (Location) float32 42kB ...
    TS_FOUND   (Location) float32 42kB ...
    TDEL       (Location) float32 42kB ...

### NEW Sampled Nature Run

<xarray.Dataset> Size: 78MB
Dimensions:    (lev: 137, Location: 10578, levp1: 138)
Coordinates:
  * lev        (lev) float64 1kB 1.0 2.0 3.0 4.0 5.0 ... 134.0 135.0 136.0 137.0
Dimensions without coordinates: Location, levp1
Data variables: (12/67)
    time       (Location) datetime64[ns] 85kB ...
    longitude  (Location) float64 85kB ...
    latitude   (Location) float64 85kB ...
    PHIS       (Location) float32 42kB ...
    HS_STDV    (Location) float32 42kB ...
    TS         (Location) float32 42kB ...
    ...         ...
    TS_FOUND   (Location) float32 42kB ...
    TDEL       (Location) float32 42kB ...
    QR         (Location, lev) float32 6MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    CO2        (Location, lev) float32 6MB 410.0 410.0 410.0 ... 410.0 410.0
    LAI        (Location, lev) float32 6MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    PLE        (Location, levp1) float32 6MB 1.0 1.825 ... 1.018e+05 1.02e+05

### GFS Surface

<xarray.Dataset> Size: 10MB
Dimensions:  (lon: 1152, lat: 721, time: 1)
Coordinates:
  * lon      (lon) float64 9kB -180.0 -179.7 -179.4 -179.1 ... 179.1 179.4 179.7
  * lat      (lat) float64 6kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * time     (time) datetime64[ns] 8B 2000-01-01
Data variables:
    stype    (time, lat, lon) float32 3MB ...
    vfrac    (time, lat, lon) float32 3MB ...
    vtype    (time, lat, lon) float32 3MB ...

In [5]:
# Instantiate RADIANCE object
# ---------------------------
gv = rd.RADIANCE(nr,sfc)
gv.populate()
gv.write(gv_file,verbose=True)
gv

[] Writing ./atms_n20_geovals.20190801T030000Z.nc4


<xarray.RADIANCE> Size: 48MB
Dimensions:                                           (nlocs: 10578,
                                                       nlevs: 137, nlevsp1: 138)
Coordinates:
    time                                              (nlocs) datetime64[ns] 85kB ...
    latitude                                          (nlocs) float64 85kB -8...
    longitude                                         (nlocs) float64 85kB 87...
    air_pressure                                      (nlocs, nlevs) float32 6MB ...
    air_pressure_levels                               (nlocs, nlevsp1) float32 6MB ...
  * nlevs                                             (nlevs) float64 1kB 1.0...
Dimensions without coordinates: nlocs, nlevsp1
Data variables: (12/39)
    stype                                             (nlocs) float32 42kB 16...
    vtype                                             (nlocs) float32 42kB 15...
    vfrac                                             (nlocs) float32 42kB 0....
    water_area_fraction                               (nlocs) float32 42kB 0....
    land_area_fraction                                (nlocs) float32 42kB ...
    ice_area_fraction                                 (nlocs) float32 42kB 0....
    ...                                                ...
    height_above_mean_sea_level_at_surface            (nlocs) float32 42kB 3....
    air_temperature                                   (nlocs, nlevs) float32 6MB ...
    humidity_mixing_ratio                             (nlocs, nlevs) float32 6MB ...
    water_vapor_mixing_ratio_wrt_dry_air              (nlocs, nlevs) float32 6MB ...
    mole_fraction_of_ozone_in_air                     (nlocs, nlevs) float32 6MB ...
    mole_fraction_of_carbon_dioxide_in_air            (nlocs, nlevs) float32 6MB ...